# Paso 1: Definición del problema:

En esta sección se identifica una necesidad específica y se transforma en un problema de Machine Learning. El conjunto de datos seleccionado debe contar, preferiblemente, con al menos 60,000 instancias y 20 variables predictoras (incluyendo una categórica) para asegurar la robustez del modelo.

-------------------------------------------------

# Paso 2: Obtencion y carga del conjunto de datos:
- El conjunto de datos se obtuvo desde la página de Kaggle, se encuentra almacenado en un formato ".csv".
"https://www.kaggle.com/datasets/zongaobian/h1b-lca-disclosure-data-2020-2024?select=Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv"



### A. Carga del conjunto de datos

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import xgboost as xgb
import optuna
import pickle
import gzip

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from pathlib import Path

/home/devian97/DATA SCIENCE N MACHINE LEARNING/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_final = pd.read_csv("df_final.csv")
col_cat = df_final.select_dtypes(include='string').columns
df_final[col_cat] = df_final[col_cat].astype('category')

df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 979113 entries, 0 to 979112
Data columns (total 20 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   PREVAILING_WAGE           979113 non-null  float64 
 1   PW_WAGE_LEVEL             920651 non-null  category
 2   EMPLOYER_STATE            978966 non-null  category
 3   WORKSITE_STATE            979113 non-null  category
 4   WORKSITE_CITY             979113 non-null  category
 5   VISA_CLASS                979113 non-null  category
 6   NAICS_CODE                979113 non-null  category
 7   FULL_TIME_POSITION        979113 non-null  category
 8   NEW_EMPLOYMENT            979113 non-null  int64   
 9   TOTAL_WORKSITE_LOCATIONS  978199 non-null  float64 
 10  WORKSITE_WORKERS          978199 non-null  float64 
 11  SECONDARY_ENTITY          979113 non-null  category
 12  H_1B_DEPENDENT            949057 non-null  category
 13  WILLFUL_VIOLATOR          949053 non-nul

# Paso 6: Construye el modelo y optimízalo

Selección y entrenamiento del algoritmo que mejor se ajuste a la naturaleza del problema. Se incluye un proceso de optimización de hiperparámetros para ajustar el modelo y alcanzar el máximo rendimiento posible en sus métricas de evaluación.

In [3]:
# Separamos las variables independientes y la variable objetivo en "x" y "y".

X = df_final.drop(columns = ['offered_anual_avg_wage'])
y = df_final['offered_anual_avg_wage']

In [4]:
# Usamos train_test_split para separar la data de testeo y de entrenamiento.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [5]:
model = xgb.XGBRegressor(
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=42
)

model.fit(X_train, y_train)

# 3. Predicción
y_pred = model.predict(X_test)

/home/devian97/DATA SCIENCE N MACHINE LEARNING/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [23:13:06] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [6]:
# Evaluar el desempeño básico
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE (Error Cuadrático Medio): {mse:,.2f}")
print(f"RMSE (Raíz del Error Cuadrático): ${rmse:,.2f}")
print(f"R2 (Coeficiente de Determinación): {r2:.4f}")

MSE (Error Cuadrático Medio): 221,591,646.20
RMSE (Raíz del Error Cuadrático): $14,885.95
R2 (Coeficiente de Determinación): 0.8621


### Generamos un dataset de prueba de 100k datos para evaluar el modelo optimizado

In [7]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

## Generamos modelo optimizado utilizando Optuna

In [8]:
def objective(trial):
    param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 3000),
        "max_depth": trial.suggest_int("max_depth", 3, 13),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0.0, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "tree_method": "hist",
        "device": "cuda",
        "enable_categorical": True,
        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 10),
        "random_state": 42,
    }

    model = xgb.XGBRegressor(**param, early_stopping_rounds=50)
    model.fit(X_train, y_train_log, eval_set=[(X_test, y_test_log)], verbose=False)

    preds = np.expm1(model.predict(X_test))
    return np.sqrt(mean_squared_error(y_test, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100)

print(f"Mejor RMSE: {study.best_value}")
print(f"Mejores parámetros: {study.best_params}")

[I 2026-05-15 23:13:06,693] A new study created in memory with name: no-name-68a806df-c3bc-4231-bf5e-5f1fb574fe58
[I 2026-05-15 23:13:16,052] Trial 0 finished with value: 16273.286220086897 and parameters: {'n_estimators': 2582, 'max_depth': 6, 'learning_rate': 0.010176387314502988, 'subsample': 0.8388289996171225, 'colsample_bytree': 0.7930012785099712, 'min_child_weight': 8, 'gamma': 4.074028639754141, 'reg_alpha': 2.4898589649920048e-05, 'reg_lambda': 0.1109410287168823, 'max_cat_to_onehot': 5}. Best is trial 0 with value: 16273.286220086897.
[I 2026-05-15 23:13:32,725] Trial 1 finished with value: 15202.674364594966 and parameters: {'n_estimators': 1555, 'max_depth': 7, 'learning_rate': 0.01701153870552376, 'subsample': 0.7996566735051985, 'colsample_bytree': 0.6751215573309148, 'min_child_weight': 18, 'gamma': 0.1910524607787023, 'reg_alpha': 4.273999007831162e-08, 'reg_lambda': 2.4392423023863247e-06, 'max_cat_to_onehot': 6}. Best is trial 1 with value: 15202.674364594966.
[I 202

Mejor RMSE: 13058.944200853206
Mejores parámetros: {'n_estimators': 2861, 'max_depth': 12, 'learning_rate': 0.03704897195372828, 'subsample': 0.8939549979228799, 'colsample_bytree': 0.5478017659654496, 'min_child_weight': 4, 'gamma': 0.0018828240095528567, 'reg_alpha': 0.06327522550130553, 'reg_lambda': 0.000112477962447909, 'max_cat_to_onehot': 3}


## Evaluamos dataset final con el estudio generado mediante Optuna

In [9]:
final_params = study.best_params.copy()

# 2. Añadimos los parámetros técnicos que no se optimizan (GPU, categorías, etc.)
final_params.update({
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "random_state": 42,
    "early_stopping_rounds": 50
})

# 3. Instanciamos el modelo usando el desempaquetado de diccionario (**)
final_model = xgb.XGBRegressor(**final_params)

# 4. Entrenamos el modelo
final_model.fit(
    X_train, y_train_log,
    eval_set=[(X_test, y_test_log)],
    verbose=100
)

[0]	validation_0-rmse:0.35524
[100]	validation_0-rmse:0.12432
[200]	validation_0-rmse:0.11679
[300]	validation_0-rmse:0.11436
[400]	validation_0-rmse:0.11311
[500]	validation_0-rmse:0.11197
[600]	validation_0-rmse:0.11122
[700]	validation_0-rmse:0.11054
[800]	validation_0-rmse:0.11004
[900]	validation_0-rmse:0.10963
[1000]	validation_0-rmse:0.10930
[1100]	validation_0-rmse:0.10895
[1200]	validation_0-rmse:0.10869
[1300]	validation_0-rmse:0.10844
[1400]	validation_0-rmse:0.10823
[1500]	validation_0-rmse:0.10801
[1600]	validation_0-rmse:0.10784
[1700]	validation_0-rmse:0.10770
[1800]	validation_0-rmse:0.10757
[1900]	validation_0-rmse:0.10746
[2000]	validation_0-rmse:0.10734
[2100]	validation_0-rmse:0.10725
[2200]	validation_0-rmse:0.10715
[2300]	validation_0-rmse:0.10707
[2400]	validation_0-rmse:0.10700
[2500]	validation_0-rmse:0.10695
[2600]	validation_0-rmse:0.10688
[2700]	validation_0-rmse:0.10683
[2800]	validation_0-rmse:0.10679
[2860]	validation_0-rmse:0.10678


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.5478017659654496
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import 

In [10]:
# 5. Evaluación final
final_preds = np.expm1(final_model.predict(X_test))
rmse_final = np.sqrt(mean_squared_error(y_test, final_preds))
r2_final = r2_score(y_test, final_preds)

print(f"Métricas del Modelo Final:")
print(f"RMSE: {rmse_final}")
print(f"R2: {r2_final}")

Métricas del Modelo Final:
RMSE: 13058.944200853206
R2: 0.8938584226100369


---------------------------

## Guardamos Modelo

In [11]:
# Ruta completa donde se guardará el modelo
categorical_cols = X_train.select_dtypes(include=["category"]).columns.tolist()
feature_names = X_train.columns.tolist()

model_bundle = {
    "model": final_model,
    "feature_names": feature_names,
    "categorical_cols": categorical_cols,
    "target": "offered_anual_avg_wage",
    "model_type": "XGBRegressor_final",
    "metrics": {
        "rmse": rmse_final,
        "r2": r2_final
    }
}

output_path = Path(
    "XGB_OPTUNA_100T_FULL_DATA.pkl.gz"
)

with gzip.open(output_path, "wb") as file:
    pickle.dump(model_bundle, file)

peso_mb = output_path.stat().st_size / (1024 ** 2)

print(f"✅ Modelo comprimido guardado correctamente en: {output_path}")
print(f"📦 Peso del archivo comprimido: {peso_mb:.2f} MB")

✅ Modelo comprimido guardado correctamente en: XGB_OPTUNA_100T_FULL_DATA.pkl.gz
📦 Peso del archivo comprimido: 153.36 MB


## Paso 7: Despliega el modelo

Fase final donde se crea una interfaz de usuario, habitualmente una aplicación web (usando Flask o Streamlit), y se implementa en una plataforma en la nube como Render o Heroku. Esto permite que el modelo sea accesible para usuarios finales o clientes potenciales.